# 06 — Evaluación del GWCCI y mapas de soja (campañas 2019–2025)

**Objetivo.** Calcular el GWCCI (Chen et al., 2023) sobre Sentinel-2 dentro de la máscara agrícola
Sentinel-2 60 m de cada campaña (notebook `05`), evaluarlo contra el Mapa Nacional de Cultivos del INTA
y generar los mapas binarios soja / no soja por departamento.

**Flujo** (cada etapa lee el producto de la anterior; lo costoso se calcula una sola vez):

| Sección | Producto | Dónde | Costo GEE |
|---|---|---|---|
| 2. GWCCI por campaña | `evaluacion_gwcci/gwcci_AAAA` (3 bandas: `feb`, `doy40_60`, `mar`; GWCCI × 10000, int16, 30 m) | asset | **alto** (medianas S2) |
| 3. Histogramas por grupo INTA | `evaluacion_gwcci/hist_gwcci_AAAA` | asset (tabla) | bajo |
| 4. Métricas (0,17 y Otsu) | `results/tables/evaluacion_gwcci_2019_2025.csv` | local | ninguno |
| 5. Superficie de soja vs INTA | `results/tables/superficie_soja_2019_2025.csv` | local | ninguno |
| 6. Mapas binarios (feb + Otsu por depto.) | `mapas_soja/soja_feb_otsu_AAAA` | asset | bajo |
| 7. Descarga de mapas por departamento | `results/maps/AAAA/*.tif` (uint8, 1 soja, 0 no soja, 255 sin dato) | local | bajo (interactivo) |
| 8. Diagnóstico: máscara por clase INTA | `evaluacion_gwcci/diag_mascara_clases_inta` | asset (tabla) | bajo |

**Convenciones.** Campaña = año de febrero (2021 = 2020/21). Grupos INTA: 1 soja (11), 2 maíz (10),
3 otros extensivos (girasol 12, algodón 15, maní 16, sorgo 18), 4 resto. Histogramas: 300 intervalos
de −0,1 a 0,5 (ancho 0,002). Umbrales: 0,17 (Chen et al., 2023) y Otsu por departamento sobre el
histograma sin etiquetas.

**Cómo usarlo.** Las celdas que lanzan tareas **no relanzan** lo que ya existe o ya está en cola, así que
se pueden volver a correr sin riesgo. En modo restringido cada tarea espera horas en cola: lanzar todo
lo que se pueda y volver más tarde.

## 1. Configuración

In [1]:
import ee, json, os, time
import numpy as np, pandas as pd

ee.Initialize(project="ee-mnolasco")

BASE = "users/mnolasco/FORMAR_2024_2026"
COL_MASCARAS = f"{BASE}/mascaras_s2_60m"                # notebook 05
CARPETA_EVAL = f"{BASE}/evaluacion_gwcci"
CARPETA_MAPAS = f"{BASE}/mapas_soja"
INTA = f"{BASE}/MNC_verano-{{}}"                        # Mapa Nacional de Cultivos (verano) por campaña

CAMPANIAS = list(range(2019, 2026))
VARIANTES = ["feb", "doy40_60", "mar"]
TABLAS = "../results/tables"
MAPAS_LOCAL = "../results/maps"

deptos = ee.FeatureCollection(f"{BASE}/departments_selected")
REGION = deptos.geometry().bounds()

for carpeta in (CARPETA_EVAL, CARPETA_MAPAS):
    try:
        ee.data.createAsset({"type": "FOLDER"}, carpeta)
    except Exception:
        pass                                            # ya existe

/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.17) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-en

In [2]:
# Utilidades para lanzar tareas sin duplicarlas
def existe(asset_id):
    try:
        ee.data.getAsset(asset_id); return True
    except Exception:
        return False

def tareas_en_cola():
    ops = ee.data.listOperations()
    return {o["metadata"].get("description") for o in ops
            if o["metadata"].get("state") in ("PENDING", "RUNNING", "READY")}

def lanzar(tarea, nombre, asset_id, en_cola):
    if existe(asset_id):
        print(f"  ya existe   {nombre}")
    elif nombre in en_cola:
        print(f"  en cola     {nombre}")
    else:
        tarea.start(); print(f"  LANZADA     {nombre}")

def estado(n=15):
    """Últimas n tareas de GEE."""
    for o in ee.data.listOperations()[:n]:
        m = o["metadata"]
        err = o.get("error", {}).get("message", "")
        print(f"{m.get('description', ''):<32} {m.get('state', ''):<10} {err}")

estado()

comparacion_mascaras_2019_2025_30m SUCCEEDED  
diag_mascara_clases_inta_30m     SUCCEEDED  
comparacion_mascaras_2019_2025_30m FAILED     Cannot overwrite asset 'projects/earthengine-legacy/assets/users/mnolasco/FORMAR_2024_2026/comparacion_mascaras_2019_2025'.
diag_mascara_clases_inta         SUCCEEDED  
soja_feb_otsu_2025               SUCCEEDED  
soja_feb_otsu_2024               SUCCEEDED  
soja_feb_otsu_2023               SUCCEEDED  
soja_feb_otsu_2022               SUCCEEDED  
soja_feb_otsu_2019               SUCCEEDED  
soja_feb_otsu_2020               SUCCEEDED  
soja_feb_otsu_2021               SUCCEEDED  
hist_gwcci_2025                  SUCCEEDED  
hist_gwcci_2024                  SUCCEEDED  
hist_gwcci_2023                  SUCCEEDED  
hist_gwcci_2022                  SUCCEEDED  


## 2. GWCCI por campaña (asset)

GWCCI = NDVI × B11 (reflectancia), con NDVI = (B8 − B4)/(B8 + B4). Se calcula sobre medianas de
Sentinel-2 SR (`COPERNICUS/S2_SR_HARMONIZED`) enmascaradas con Cloud Score+ (cs_cdf ≥ 0,60), en tres
ventanas: febrero, DOY 40–60 (ventana de Chen et al.) y marzo. Se enmascara con la máscara S2 60 m de la
misma campaña y se guarda × 10000 como int16 a 30 m. **Es la única etapa costosa.**

In [3]:
S2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
CS = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

def mascara(c):
    return ee.ImageCollection(COL_MASCARAS).filter(ee.Filter.eq("campania", c)).mosaic().eq(1)

def mediana_gwcci(ini, fin, nombre):
    col = (S2.filterBounds(REGION).filterDate(ini, fin)
             .linkCollection(CS, ["cs_cdf"])
             .map(lambda i: i.updateMask(i.select("cs_cdf").gte(0.60))))
    r = col.select(["B4", "B8", "B11"]).median().divide(10000)
    return r.normalizedDifference(["B8", "B4"]).multiply(r.select("B11")).rename(nombre)

def gwcci_campania(c):
    ene1 = ee.Date.fromYMD(c, 1, 1)
    g = ee.Image.cat(
        mediana_gwcci(f"{c}-02-01", f"{c}-03-01", "feb"),
        mediana_gwcci(ene1.advance(39, "day"), ene1.advance(60, "day"), "doy40_60"),
        mediana_gwcci(f"{c}-03-01", f"{c}-04-01", "mar"))
    return g.updateMask(mascara(c)).multiply(10000).round().toInt16()

In [4]:
# Lanzar (omite lo que ya existe o está en cola)
en_cola = tareas_en_cola()
for c in CAMPANIAS:
    nombre = f"gwcci_{c}"
    asset = f"{CARPETA_EVAL}/{nombre}"
    t = ee.batch.Export.image.toAsset(
        image=gwcci_campania(c), description=nombre, assetId=asset,
        region=REGION, scale=30, crs="EPSG:32720",
        pyramidingPolicy={".default": "mean"}, maxPixels=1e13)
    lanzar(t, nombre, asset, en_cola)

  ya existe   gwcci_2019
  ya existe   gwcci_2020
  ya existe   gwcci_2021
  ya existe   gwcci_2022
  ya existe   gwcci_2023
  ya existe   gwcci_2024
  ya existe   gwcci_2025


## 3. Histogramas del GWCCI por departamento y grupo INTA (asset)

Para cada departamento se calcula el histograma del GWCCI de las tres variantes, separado por grupo
INTA. Con ellos se obtienen localmente las métricas para cualquier umbral. Las tablas de assets no
admiten listas de diccionarios, por lo que cada histograma se guarda como conteos (300 valores) y el
bloque completo como texto JSON (`compactar`).

*Nota:* el reductor agrupado descarta el píxel si alguna de las tres variantes no tiene dato; las tres
variantes se evalúan sobre la misma superficie.

In [4]:
REDUCTOR_INT = (ee.Reducer.fixedHistogram(-1000, 5000, 300).repeat(3)   # GWCCI × 10000
                .group(groupField=3, groupName="grupo"))

def grupos_inta(c):
    return (ee.Image(INTA.format(c)).select(0)
            .remap([11, 10, 12, 15, 16, 18], [1, 2, 3, 3, 3, 3], 4).rename("grupo"))

def compactar(f):
    def un_grupo(g):
        d = ee.Dictionary(g)
        conteos = ee.List(d.get("histogram")).map(
            lambda a: ee.Algorithms.If(a, ee.Array(a).slice(1, 1, 2).project([0]).toList(), []))
        return ee.Dictionary({"grupo": d.get("grupo"), "h": conteos})
    return f.set("groups", ee.String.encodeJSON(ee.List(f.get("groups")).map(un_grupo)))

def hist_desde_asset(c):
    g = ee.Image(f"{CARPETA_EVAL}/gwcci_{c}")
    return (g.addBands(grupos_inta(c))
            .reduceRegions(deptos, REDUCTOR_INT, 30, "EPSG:32720", tileScale=8)
            .map(compactar)
            .map(lambda f: f.set("campania", c).setGeometry(f.geometry().centroid(100))))

In [5]:
# Lanzar para las campañas cuyo GWCCI ya existe
en_cola = tareas_en_cola()
for c in CAMPANIAS:
    if not existe(f"{CARPETA_EVAL}/gwcci_{c}"):
        print(f"  falta gwcci_{c}"); continue
    nombre = f"hist_gwcci_{c}"
    asset = f"{CARPETA_EVAL}/{nombre}"
    lanzar(ee.batch.Export.table.toAsset(hist_desde_asset(c), nombre, asset), nombre, asset, en_cola)

  ya existe   hist_gwcci_2019
  ya existe   hist_gwcci_2020
  ya existe   hist_gwcci_2021
  ya existe   hist_gwcci_2022
  ya existe   hist_gwcci_2023
  ya existe   hist_gwcci_2024
  ya existe   hist_gwcci_2025


## 4. Métricas por departamento (local)

Para cada departamento, campaña, variante y umbral (0,17 y Otsu) se calculan, dentro de la máscara:
kappa, exactitud global (OA), exactitud del productor (PA) y del usuario (UA) para la clase soja, y error
de área (pp del área de la máscara). El umbral de Otsu se obtiene del histograma total (sin etiquetas),
maximizando la varianza entre clases. Se procesan las campañas disponibles.

In [6]:
BORDES = np.linspace(-0.1, 0.5, 301)
CENTROS = (BORDES[:-1] + BORDES[1:]) / 2
K017 = int(np.searchsorted(BORDES, 0.17 - 1e-9))     # primer intervalo ≥ 0,17

def otsu_k(h):
    """Índice del primer intervalo clasificado como soja (umbral de Otsu)."""
    p = h / h.sum(); w = np.cumsum(p); mu = np.cumsum(p * CENTROS)
    with np.errstate(divide="ignore", invalid="ignore"):
        s2 = (mu[-1] * w - mu) ** 2 / (w * (1 - w))
    s2[~np.isfinite(s2)] = 0
    return int(np.argmax(s2)) + 1

def metricas(soja, otro, k):
    TP, FN = soja[k:].sum(), soja[:k].sum()
    FP, TN = otro[k:].sum(), otro[:k].sum()
    N = TP + FN + FP + TN
    if N == 0 or TP + FN == 0:
        return None
    oa = (TP + TN) / N
    pe = ((TP + FN) * (TP + FP) + (FP + TN) * (FN + TN)) / N**2
    return dict(kappa=(oa - pe) / (1 - pe), OA=oa, PA=TP / (TP + FN),
                UA=TP / (TP + FP) if TP + FP else np.nan,
                err_area_pp=100 * ((TP + FP) - (TP + FN)) / N,
                soja_ref_pct=100 * (TP + FN) / N, ha=N * 0.09)

filas = []
for c in CAMPANIAS:
    asset = f"{CARPETA_EVAL}/hist_gwcci_{c}"
    if not existe(asset):
        print("todavía no disponible:", c); continue
    df = ee.data.computeFeatures({"expression": ee.FeatureCollection(asset),
                                  "fileFormat": "PANDAS_DATAFRAME"})
    for _, r in df.iterrows():
        grupos = {int(g["grupo"]): g["h"] for g in json.loads(r["groups"])}
        for i, v in enumerate(VARIANTES):
            H = {gr: np.array(h[i], float) if len(h[i]) else np.zeros(300) for gr, h in grupos.items()}
            soja = H.get(1, np.zeros(300))
            otro = sum((H[g] for g in H if g != 1), np.zeros(300))
            if (soja + otro).sum() == 0:
                continue
            for umbral, k in [("0.17", K017), ("otsu", otsu_k(soja + otro))]:
                m = metricas(soja, otro, k)
                if m:
                    filas.append(dict(campania=c, link=r["link"], departamento=r["departamen"],
                                      provincia=r["provincia"], variante=v, umbral=umbral,
                                      corte=round(BORDES[k], 3), **m))

ev = pd.DataFrame(filas)
ev.to_csv(f"{TABLAS}/evaluacion_gwcci_2019_2025.csv", index=False)
ev["abs_err_pp"] = ev.err_area_pp.abs()

print("== Promedio de departamentos: campaña × variante × umbral ==")
print(ev.groupby(["campania", "variante", "umbral"])[["kappa", "err_area_pp", "abs_err_pp", "corte"]]
        .mean().round(3).unstack("umbral").to_string())
print("\n== Kappa por provincia (campañas disponibles) ==")
print(ev.groupby(["provincia", "variante", "umbral"]).kappa.mean().round(3)
        .unstack(["variante", "umbral"]).to_string())

/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/dat

== Promedio de departamentos: campaña × variante × umbral ==
                   kappa        err_area_pp        abs_err_pp         corte       
umbral              0.17   otsu        0.17   otsu       0.17    otsu  0.17   otsu
campania variante                                                                 
2019     doy40_60  0.509  0.486       0.862  9.405     10.695  13.611  0.17  0.159
         feb       0.509  0.495       1.802  7.320     10.857  12.370  0.17  0.164
         mar       0.359  0.380     -14.535  5.966     16.547  13.202  0.17  0.145
2020     doy40_60  0.512  0.620     -16.131  1.309     16.438   6.955  0.17  0.153
         feb       0.544  0.619     -13.624 -1.288     14.049   7.086  0.17  0.158
         mar       0.236  0.455     -39.684 -2.631     39.684   9.702  0.17  0.133
2021     doy40_60  0.434  0.551     -23.111 -2.472     23.138  11.875  0.17  0.153
         feb       0.467  0.550     -20.308 -4.207     20.367  11.721  0.17  0.156
         mar       0.299  

## 5. Superficie de soja por departamento vs INTA (local)

Las métricas de la sección 4 se calculan dentro de la máscara y no ven la soja que la máscara omite.
Aquí se compara la superficie de soja estimada por el GWCCI con **toda** la soja del INTA del
departamento (tabla `comparacion_mascaras_2019_2025.csv`, notebook `05`), que es la magnitud que informa
la estadística oficial. *Nota:* la soja del INTA proviene de la comparación a 60 m.

In [9]:
asset_30 = f"{BASE}/comparacion_mascaras_2019_2025_30m"
print("asset a 30 m disponible:", existe(asset_30))

cmp30 = ee.data.computeFeatures({"expression": ee.FeatureCollection(asset_30),
                                 "fileFormat": "PANDAS_DATAFRAME"})
cmp30 = cmp30.drop(columns=["geo"], errors="ignore")
cmp30.to_csv(f"{TABLAS}/comparacion_mascaras_2019_2025.csv", index=False)

# Control: la soja del INTA a 30 m tiene que ser MAYOR que la de 60 m (antes, 2021 = 12.351 kha)
print((cmp30.groupby("campania").inta_soja.sum() / 1e3).round(1))

/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


asset a 30 m disponible: True
campania
2019   10870.4
2020   12831.7
2021   13014.8
2022   10687.8
2023   10855.7
2024   11635.0
2025   13737.5
Name: inta_soja, dtype: float64


In [10]:
cmp = pd.read_csv(f"{TABLAS}/comparacion_mascaras_2019_2025.csv", dtype={"link": str})
ev = pd.read_csv(f"{TABLAS}/evaluacion_gwcci_2019_2025.csv", dtype={"link": str})

ev["soja_est_ha"] = ev.ha * (ev.soja_ref_pct + ev.err_area_pp) / 100
sup = ev.merge(cmp[["link", "campania", "inta_soja", "s2_soja"]], on=["link", "campania"])
sup.to_csv(f"{TABLAS}/superficie_soja_2019_2025.csv", index=False)

def resumen(g):
    est, ref = g.soja_est_ha.sum(), g.inta_soja.sum()
    return pd.Series({"soja_INTA_kha": ref / 1e3, "soja_GWCCI_kha": est / 1e3,
                      "error_total_%": 100 * (est - ref) / ref,
                      "omitida_por_mascara_%": 100 * (1 - g.s2_soja.sum() / ref),
                      "err_abs_mediano_deptos_%": (100 * (g.soja_est_ha - g.inta_soja).abs() / g.inta_soja).median()})

pd.set_option("display.float_format", "{:.1f}".format)
print("== Superficie de soja: GWCCI vs INTA (total del departamento) ==")
print(sup.groupby(["campania", "variante", "umbral"]).apply(resumen).to_string())
print("\n== Febrero + Otsu, por provincia ==")
print(sup[(sup.variante == "feb") & (sup.umbral == "otsu")]
        .groupby(["provincia", "campania"]).apply(resumen).to_string())

== Superficie de soja: GWCCI vs INTA (total del departamento) ==
                          soja_INTA_kha  soja_GWCCI_kha  error_total_%  omitida_por_mascara_%  err_abs_mediano_deptos_%
campania variante umbral                                                                                               
2019     doy40_60 0.17          10870.4         10145.4           -6.7                    9.1                      15.5
                  otsu          10870.4         11400.1            4.9                    9.1                      16.5
         feb      0.17          10870.4         10324.2           -5.0                    9.1                      16.3
                  otsu          10870.4         11056.6            1.7                    9.1                      16.5
         mar      0.17          10870.4          6913.8          -36.4                    9.1                      36.5
                  otsu          10870.4         10515.6           -3.3                    9.1  

## 6. Mapas binarios soja / no soja (asset)

Variante elegida: **mediana de febrero + umbral de Otsu por departamento** (mismo rendimiento que la
ventana de Chen, más simple y disponible a inicios de marzo). Se aplica a la banda `feb` del asset del
GWCCI el corte de Otsu de cada departamento (sección 4). Resultado uint8: 1 soja, 0 no soja, sin dato
fuera de la máscara. Requiere que la sección 4 se haya corrido con la campaña.

In [8]:
ev = pd.read_csv(f"{TABLAS}/evaluacion_gwcci_2019_2025.csv", dtype={"link": str})
SEL = ev[(ev.variante == "feb") & (ev.umbral == "otsu")]

def imagen_umbral(c):
    u = SEL[SEL.campania == c].set_index("link").corte.mul(10000).round().astype(int).to_dict()
    dic = ee.Dictionary(u)
    fc = (deptos.filter(ee.Filter.inList("link", list(u.keys())))
                .map(lambda f: f.set("u", dic.get(f.get("link")))))
    return fc.reduceToImage(["u"], ee.Reducer.first())

def mapa_soja(c):
    g = ee.Image(f"{CARPETA_EVAL}/gwcci_{c}").select("feb")
    return (g.gte(imagen_umbral(c)).toUint8().rename("soja")
             .set({"campania": c, "variante": "feb", "umbral": "otsu_depto"}))

en_cola = tareas_en_cola()
for c in sorted(int(x) for x in SEL.campania.unique()):   # int nativo: GEE no acepta numpy.int64
    nombre = f"soja_feb_otsu_{c}"
    asset = f"{CARPETA_MAPAS}/{nombre}"
    t = ee.batch.Export.image.toAsset(
        image=mapa_soja(c), description=nombre, assetId=asset,
        region=REGION, scale=30, crs="EPSG:32720",
        pyramidingPolicy={"soja": "mode"}, maxPixels=1e13)
    lanzar(t, nombre, asset, en_cola)

/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


  ya existe   soja_feb_otsu_2019


/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


  ya existe   soja_feb_otsu_2020


/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


...

/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


  ya existe   soja_feb_otsu_2023


/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


  ya existe   soja_feb_otsu_2024
  ya existe   soja_feb_otsu_2025


/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


## 7. Descarga de los mapas por departamento (local)

Descarga interactiva (no pasa por la cola de tareas). Si el asset del mapa todavía no existe, el mapa se
calcula en el momento desde el GWCCI y los umbrales (sección 6), lo que da el mismo resultado.
Salida: `results/maps/AAAA/soja_feb_otsu_AAAA_<link>.tif`, uint8, 1 = soja, 0 = no soja,
255 = sin dato (fuera de la máscara o del departamento), compresión DEFLATE.

In [9]:
import requests, math
import rasterio
from rasterio.transform import Affine

MAX_PX = 20_000_000          # GEE limita cada descarga a ~  MB; uint8 = 1 byte por píxel

def _bajar_bloque(img, ancho, alto, x0, y0):
    """Descarga un bloque alineado a la grilla de 30 m (EPSG:32720) y lo devuelve como array."""
    url = img.getDownloadURL({"dimensions": f"{ancho}x{alto}", "crs": "EPSG:32720",
                              "crs_transform": [30, 0, x0, 0, -30, y0], "format": "GEO_TIFF"})
    r = requests.get(url, timeout=600); r.raise_for_status()
    with rasterio.MemoryFile(r.content) as mf, mf.open() as src:
        return src.read(1)

def descargar_depto(mapa, link, salida, reintentos=3):
    """Descarga el mapa de un departamento. Si GEE rechaza el tamaño del pedido, divide el bloque
    en dos franjas (recursivamente) hasta que entre en el límite."""
    geom = deptos.filter(ee.Filter.eq("link", link)).geometry()
    xs, ys = zip(*geom.bounds(1, "EPSG:32720").coordinates().get(0).getInfo())
    x0 = math.floor(min(xs) / 30) * 30; y0 = math.ceil(max(ys) / 30) * 30
    W = math.ceil((max(xs) - x0) / 30); H = math.ceil((y0 - min(ys)) / 30)
    img = mapa.clip(geom).unmask(255).toUint8()                       # 255 = sin dato
    arr = np.full((H, W), 255, np.uint8)

    def bajar(f0, h):
        for intento in range(reintentos):
            try:
                arr[f0:f0 + h] = _bajar_bloque(img, W, h, x0, y0 - 30 * f0); return
            except Exception as e:
                if "request size" in str(e) and h > 1:                 # demasiado grande: partir
                    mitad = h // 2
                    bajar(f0, mitad); bajar(f0 + mitad, h - mitad); return
                if intento == reintentos - 1:
                    raise
                print(f"  {link}: reintento ({str(e)[:70]})"); time.sleep(20)

    paso = math.ceil(H / max(1, math.ceil(W * H / MAX_PX)))
    for f0 in range(0, H, paso):
        bajar(f0, min(paso, H - f0))
    perfil = dict(driver="GTiff", height=H, width=W, count=1, dtype="uint8", crs="EPSG:32720",
                  transform=Affine(30, 0, x0, 0, -30, y0), nodata=255, compress="deflate", tiled=True)
    with rasterio.open(salida, "w", **perfil) as dst:
        dst.write(arr, 1)

def descargar_mapas(c):
    asset = f"{CARPETA_MAPAS}/soja_feb_otsu_{c}"
    mapa = ee.Image(asset) if existe(asset) else mapa_soja(c)
    carpeta = f"{MAPAS_LOCAL}/{c}"; os.makedirs(carpeta, exist_ok=True)
    fallidos = []
    for link in SEL[SEL.campania == c].link.unique():
        salida = f"{carpeta}/soja_feb_otsu_{c}_{link}.tif"
        if os.path.exists(salida):
            continue
        try:
            descargar_depto(mapa, link, salida)
        except Exception as e:
            fallidos.append(link); print(f"  {link}: FALLÓ ({str(e)[:80]})")
    print(f"{c}: {len(os.listdir(carpeta))} archivos en {carpeta}; fallidos: {fallidos or 'ninguno'}")

def comprimir(carpeta):
    """Reescribe los GeoTIFF con compresión DEFLATE y nodata = 255 (requiere rasterio)."""
    import rasterio, glob
    for f in glob.glob(f"{carpeta}/*.tif"):
        with rasterio.open(f) as src:
            perfil = src.profile; datos = src.read()
        perfil.update(compress="deflate", nodata=255, dtype="uint8")
        with rasterio.open(f, "w", **perfil) as dst:
            dst.write(datos)

In [16]:
# Descargar (los archivos nuevos ya salen comprimidos; comprimir() sirve para los descargados antes)
for c in [2025]:
    descargar_mapas(c); comprimir(f"{MAPAS_LOCAL}/{c}")

2025: 95 archivos en ../results/maps/2025; fallidos: ninguno


## 8. Diagnóstico: composición de la máscara por clase INTA (asset)

Superficie (ha) de la máscara S2 60 m por clase del Mapa Nacional de Cultivos, por departamento y campaña.
Sirve para explicar la sobreestimación del área agrícola (La Pampa, campañas 2019 y 2023): qué clases del
INTA (pasturas, barbecho, no agrícola, etc.) ocupan la superficie de la máscara que no es agricultura
extensiva. Una sola tarea para las 7 campañas, a 60 m.

In [7]:
def area_por_clase(c):
    img = (ee.Image.pixelArea().divide(1e4)
             .addBands(ee.Image(INTA.format(c)).select(0).rename("clase"))
             .updateMask(mascara(c)))
    return (img.reduceRegions(deptos, ee.Reducer.sum().group(groupField=1, groupName="clase"),
                              30, "EPSG:32720", tileScale=8)
               .map(lambda f: f.set("groups", ee.String.encodeJSON(f.get("groups")), "campania", c)
                               .setGeometry(f.geometry().centroid(100))))

nombre = "diag_mascara_clases_inta_30m"
asset = f"{CARPETA_EVAL}/{nombre}"
fc = ee.FeatureCollection([area_por_clase(c) for c in CAMPANIAS]).flatten()
lanzar(ee.batch.Export.table.toAsset(fc, nombre, asset), nombre, asset, tareas_en_cola())

  LANZADA     diag_mascara_clases_inta_30m


In [8]:
CLASES = {10: "Maíz", 11: "Soja", 12: "Girasol", 13: "Poroto", 14: "Caña de azúcar", 15: "Algodón",
          16: "Maní", 17: "Arroz", 18: "Sorgo", 19: "Girasol-CV", 21: "Barbecho", 22: "No agrícola",
          26: "Papa", 27: "Verdeo de maíz", 28: "Verdeo de sorgo", 30: "Tabaco"}
EXTENSIVOS = {10, 11, 12, 15, 16, 18}

df = ee.data.computeFeatures({"expression": ee.FeatureCollection(f"{CARPETA_EVAL}/diag_mascara_clases_inta_30m"),
                              "fileFormat": "PANDAS_DATAFRAME"})
filas = [dict(campania=r["campania"], link=r["link"], provincia=r["provincia"],
              clase=int(g["clase"]), ha=g["sum"])
         for _, r in df.iterrows() for g in json.loads(r["groups"])]
diag = pd.DataFrame(filas)
diag["nombre"] = diag.clase.map(CLASES).fillna("código " + diag.clase.astype(str))
diag["tipo"] = np.where(diag.clase.isin(EXTENSIVOS), "Agricultura extensiva", diag.nombre)
diag.to_csv(f"{TABLAS}/diag_mascara_clases_inta_2019_2025.csv", index=False)

def composicion(por):
    t = diag.groupby(por + ["tipo"]).ha.sum().unstack("tipo").fillna(0)
    t = 100 * t.div(t.sum(axis=1), axis=0)                       # % de la superficie de la máscara
    orden = ["Agricultura extensiva"] + [c for c in t.sum().sort_values(ascending=False).index
                                         if c != "Agricultura extensiva"]
    return t[orden[:7]].round(1)                                  # extensiva + las 6 clases más grandes

pd.set_option("display.width", 160)
print("== Composición de la máscara (% de su superficie) por campaña ==")
print(composicion(["campania"]).to_string())
print("\n== Por provincia (todas las campañas) ==")
print(composicion(["provincia"]).to_string())
print("\n== La Pampa, por campaña ==")
print(composicion(["provincia", "campania"]).loc["La Pampa"].to_string())

/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


== Composición de la máscara (% de su superficie) por campaña ==
tipo      Agricultura extensiva  No agrícola  Barbecho  código 255  Arroz  Papa  Girasol-CV
campania                                                                                   
2019                       78.3         18.8       2.5         0.0    0.0   0.0         0.4
2020                       90.0          8.8       1.1         0.0    0.1   0.0         0.0
2021                       92.9          6.0       0.9         0.0    0.1   0.0         0.0
2022                       89.7          9.0       1.1         0.0    0.1   0.1         0.0
2023                       81.8         16.8       0.7         0.5    0.1   0.1         0.0
2024                       90.7          7.7       1.0         0.4    0.1   0.1         0.0
2025                       90.0          8.4       0.9         0.4    0.1   0.2         0.0

== Por provincia (todas las campañas) ==
tipo          Agricultura extensiva  No agrícola  Barbecho  Arroz

In [12]:
for c in CAMPANIAS:
    a = ee.data.getAsset(INTA.format(c))
    escala = ee.Image(INTA.format(c)).projection().nominalScale().getInfo()
    politica = [b.get("pyramidingPolicy", "MEAN (por defecto)") for b in a.get("bands", [])]
    print(c, f"resolución nativa: {escala:.0f} m", "| pirámides:", politica)

# Prueba directa en un departamento: clases a resolución nativa vs a 60 m
geom = deptos.filter(ee.Filter.eq("link", "06147")).geometry()          # Carlos Casares
inta = ee.Image(INTA.format(2021)).select(0)
nativa = inta.reduceRegion(ee.Reducer.frequencyHistogram(), geom, inta.projection().nominalScale(), maxPixels=1e9)
a60 = inta.reduceRegion(ee.Reducer.frequencyHistogram(), geom, 60, "EPSG:32720", maxPixels=1e9)
print("nativa:", nativa.getInfo())
print("a 60 m:", a60.getInfo())

/home/jovyan/.conda/envs/formar-gwcci/lib/python3.11/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


2019 resolución nativa: 30 m | pirámides: ['MEAN']
2020 resolución nativa: 30 m | pirámides: ['MEAN']
2021 resolución nativa: 30 m | pirámides: ['MEAN']
2022 resolución nativa: 30 m | pirámides: ['MEAN']
2023 resolución nativa: 30 m | pirámides: ['MEAN']
2024 resolución nativa: 30 m | pirámides: ['MEAN']
2025 resolución nativa: 30 m | pirámides: ['MEAN']
nativa: {'b1': {'10': 576110.9019607845, '11': 891639.5921568627, '12': 1952, '18': 94, '22': 1666280.2823529413}}
a 60 m: {'b1': {'10': 94133.79999999992, '11': 161644.6392156862, '12': 301.5019607843137, '13': 12695.435294117644, '14': 15947.894117647076, '15': 57.44313725490199, '16': 13180.733333333352, '17': 13484.78431372549, '18': 301.12156862745127, '19': 25740.000000000015, '20': 12, '22': 297276.73333332967}}


## Registro de resultados

- **2021 (feb + Otsu):** kappa 0,550 dentro de la máscara S2 (0,546 con AgriLand-2019): el kappa dentro
  de la máscara casi no cambia porque no ve la soja omitida por la máscara.
- **Superficie total (feb + Otsu):** −5,6 % (2020) y −10,9 % (2021) frente al INTA; la omisión de la
  máscara (7–9 %) explica la mayor parte. Con AgriLand-2019 (22 % de omisión) el sesgo habría sido de
  ~−22 a −24 % (estimación).
- **Otsu vs 0,17:** Otsu mejora en 2020–2021 (+0,08 a +0,11 de kappa; cortes medios 0,153–0,164); en
  2019 es mejor 0,17 (referencia INTA 2019 atípica).
- **Provincias:** Entre Ríos ±6 % de superficie; Santa Fe −6 a −15 %; Córdoba subestima (−8 a −17 %);
  La Pampa sobreestima (+30 %).
- Campañas 2022–2025: pendientes.